# 04 - SQL Analytics

## Objective
Answer portfolio-level business questions in SQL, kept deliberately separate from
the Python feature code: the analytics layer describes the book, the Python layer
prepares model inputs.

Engine: **DuckDB** over the processed CSV, so the whole analysis reproduces with
no database server.

In [1]:
import sys
from pathlib import Path

# Make `src` importable when the notebook is run from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [2]:
import duckdb
import pandas as pd

from src import config
from src.analytics import runner

## Configuration and data loading

In [3]:
SQL_FILE = config.SQL_DIR / "analytics.sql"
loans = pd.read_csv(config.FEATURED_DATA_FILE)
connection = duckdb.connect(database=":memory:")
connection.register("loans", loans)
print(f"{len(loans):,} loans registered as the `loans` table")

32,416 loans registered as the `loans` table


## The query catalogue

Queries live in `sql/analytics.sql` and are split on their `-- name:` headers.

In [4]:
queries = runner.parse_queries(SQL_FILE)
list(queries)

['01_portfolio_overview',
 '02_default_rate_by_employment',
 '03_default_rate_by_income_group',
 '04_default_rate_by_loan_intent',
 '05_default_rate_by_credit_grade',
 '06_default_rate_by_home_ownership',
 '07_default_rate_by_age_group',
 '08_default_rate_by_prior_default',
 '09_loan_burden_bands',
 '10_high_risk_customers',
 '11_portfolio_distribution_by_loan_size',
 '12_worst_segments_grade_x_intent']

## Portfolio KPIs

In [5]:
connection.execute(queries["01_portfolio_overview"]).fetchdf()

,total_customers,total_loans,default_rate_pct,avg_loan_amount,avg_income,total_exposure,defaulted_exposure
0,32416,32416,21.87,9593.85,65908.58,310994100.0,76968675.0


## Risk by customer characteristics

In [6]:
for name in ["02_default_rate_by_employment", "03_default_rate_by_income_group",
             "06_default_rate_by_home_ownership", "07_default_rate_by_age_group"]:
    print(f"--- {name} ---")
    display(connection.execute(queries[name]).fetchdf())

--- 02_default_rate_by_employment ---


,employment_band,customers,default_rate_pct,avg_loan_amount
0,Unknown,889,31.72,7105.17
1,<1 year,4086,27.97,8556.28
2,1-3 years,6729,26.60,9122.99
3,3-6 years,9231,20.05,9556.04
4,6-10 years,8565,18.10,10270.58
5,10+ years,2916,16.22,11024.93


--- 03_default_rate_by_income_group ---


,income_group,customers,default_rate_pct,avg_loan_amount,avg_loan_to_income_pct
0,<30k,4486,45.59,5453.75,23.21
1,30k-60k,14136,23.76,8428.28,18.72
2,60k-100k,9603,13.38,11203.94,14.55
3,150k+,1131,9.81,16321.00,7.97
4,100k-150k,3059,9.45,13511.26,11.24
5,None,1,0.00,5000.00,0.00


--- 06_default_rate_by_home_ownership ---


,person_home_ownership,customers,default_rate_pct,avg_income
0,RENT,16378,31.61,55019.63
1,OTHER,106,31.13,76561.27
2,MORTGAGE,13369,12.62,80708.22
3,OWN,2563,7.49,57858.83


--- 07_default_rate_by_age_group ---


,age_group,customers,default_rate_pct,avg_credit_history_years
0,18-25,15245,23.14,3.00
1,26-35,13711,20.69,6.69
2,36-45,2809,20.68,13.92
3,46-60,581,21.86,17.63
4,60+,65,26.15,24.31


## Risk by product and pricing

In [7]:
for name in ["04_default_rate_by_loan_intent", "05_default_rate_by_credit_grade",
             "08_default_rate_by_prior_default", "09_loan_burden_bands"]:
    print(f"--- {name} ---")
    display(connection.execute(queries[name]).fetchdf())

--- 04_default_rate_by_loan_intent ---


,loan_intent,loans,default_rate_pct,avg_loan_amount,total_exposure
0,DEBTCONSOLIDATION,5189,28.68,9605.27,49841750.0
1,MEDICAL,6042,26.76,9271.58,56018875.0
2,HOMEIMPROVEMENT,3594,26.15,10362.12,37241475.0
3,PERSONAL,5498,19.90,9579.68,52669100.0
4,EDUCATION,6411,17.25,9480.05,60776600.0
5,VENTURE,5682,14.85,9582.24,54446300.0


--- 05_default_rate_by_credit_grade ---


,loan_grade,loans,default_rate_pct,avg_interest_rate,avg_loan_amount
0,A,10703,9.96,7.33,8545.70
1,B,10387,16.32,11.00,9992.23
2,C,6438,20.75,13.46,9219.18
3,D,3620,59.06,15.36,10855.76
4,E,963,64.49,17.01,12919.91
5,F,241,70.54,18.61,14717.32
6,G,64,98.44,20.25,17195.70


--- 08_default_rate_by_prior_default ---


,bureau_status,customers,default_rate_pct,avg_interest_rate
0,Prior default on file,5730,37.87,14.52
1,Clean bureau file,26686,18.43,10.26


--- 09_loan_burden_bands ---


,loan_to_income_band,loans,default_rate_pct,avg_loan_amount
0,<10%,8907,11.59,4971.20
1,10-20%,12456,14.74,9186.05
2,20-30%,6786,20.97,12655.03
3,30-40%,2934,61.93,14850.57
4,40%+,1333,73.59,17138.48


## Portfolio concentration and watchlist

In [8]:
for name in ["10_high_risk_customers", "11_portfolio_distribution_by_loan_size",
             "12_worst_segments_grade_x_intent"]:
    print(f"--- {name} ---")
    display(connection.execute(queries[name]).fetchdf())

--- 10_high_risk_customers ---


,high_risk_customers,pct_of_portfolio,default_rate_pct,exposure_at_risk
0,1292,3.99,76.16,21673950.0


--- 11_portfolio_distribution_by_loan_size ---


,loan_amount_group,loans,pct_of_loans,total_exposure,default_rate_pct
0,5k-10k,11983,36.97,94293250.0,18.04
1,15k-25k,4462,13.76,90638200.0,32.27
2,10k-15k,6091,18.79,79091175.0,23.08
3,<5k,9430,29.09,32773875.0,20.47
4,25k+,450,1.39,14197600.0,33.56


--- 12_worst_segments_grade_x_intent ---


,loan_grade,loan_intent,loans,default_rate_pct,total_exposure
0,E,DEBTCONSOLIDATION,144,100.00,1810950.0
1,E,MEDICAL,167,95.81,2158625.0
2,D,DEBTCONSOLIDATION,598,92.81,6415075.0
3,D,MEDICAL,655,87.94,6973675.0
4,D,HOMEIMPROVEMENT,487,54.00,5546300.0
5,E,VENTURE,177,51.98,2242625.0
6,E,HOMEIMPROVEMENT,143,51.75,2022725.0
7,E,PERSONAL,147,48.30,1845400.0
8,D,PERSONAL,643,43.55,6998750.0
9,E,EDUCATION,185,43.24,2361550.0


## Ad-hoc query

The same connection supports free-form analysis.

In [9]:
connection.execute("""
    SELECT loan_grade,
           COUNT(*)                         AS loans,
           ROUND(AVG(loan_int_rate), 2)     AS avg_rate,
           ROUND(AVG(loan_percent_income), 3) AS avg_burden,
           ROUND(AVG(loan_status) * 100, 2) AS default_rate_pct
    FROM loans
    WHERE person_income < 40000
    GROUP BY 1
    ORDER BY 1
""").fetchdf()

,loan_grade,loans,avg_rate,avg_burden,default_rate_pct
0,A,2518,7.40,0.223,27.44
1,B,2747,10.96,0.223,33.13
2,C,1869,13.45,0.219,37.51
3,D,1131,15.34,0.220,75.69
4,E,240,17.02,0.237,82.92
5,F,53,19.20,0.230,77.36
6,G,10,20.27,0.270,100.00


## Results

The full set of results is also written to `reports/sql_analytics.md` by `run_pipeline.py`.

In [10]:
results = runner.run_queries(loans, SQL_FILE)
report_path = runner.save_report(results)
print("Report written to", report_path)
connection.close()

Report written to D:\Downloads\ML-Driven-Credit-Risk-Loan-Default-Prediction-System-main\credit-risk-loan-default-ml\reports\sql_analytics.md


## Interpretation
The SQL layer confirms, at portfolio level, the same three drivers the EDA found:
affordability (loan-to-income band), pricing/grade, and stability (employment,
bureau history). It also quantifies *exposure*, which the modelling metrics do not:
a segment can have a moderate default rate and still dominate expected loss because
its average ticket is large.

## Conclusion
These queries are what a risk or portfolio analyst would run monthly; the model in
notebook 05 turns the same signals into a customer-level probability.